[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/spin_groups/spin_groups.ipynb)

# The Spin Groups

In every signature a rotor is the exponential of a bivector, and the rotors of a signature form its spin group. How many of a group's generators rotate and how many boost, and whether its spinors split in two or are complex, can be read off three maps built by leaving slots open. Every signature up to six dimensions fits in one algebra with six directions that square to plus one and three that square to minus one; choosing directions chooses the signature.

The rotations of space and the Lorentz transformations of spacetime are two of these groups, of the signatures (3, 0) and (3, 1).

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from functools import reduce

import numpy as np
from IPython.display import Image, display

from numga import Algebra, NumpyContext
from examples.animation import save_animation
from examples.math.spin_groups import render

np.set_printoptions(precision=4, suppress=True)

# Six directions squaring to plus one and three squaring to minus one.
ga = Algebra("x+y+z+w+v+u+t-s-r-")
context = NumpyContext(ga)
mv = context.multivector
one = mv.scalar([1.0])                                                          # [] Scalar


def signature(p: int, q: int):
    """The first p positive and q negative directions: their bivectors and even multivectors as
    types, the product of the positive ones, and their pseudoscalar."""
    names = "xyzwvu"[:p] + "tsr"[:q]
    closure = ga.gatype(ga.subspace(" ".join(names))).minimal_subalgebra.output_subspace
    Bivector = ga.gatype(closure.intersection(ga.subspace.bivector()))
    Even = ga.gatype(closure.intersection(ga.subspace.even()))
    positives = reduce(lambda product, name: product * getattr(mv, name), "xyzwvu"[:p], one)
    pseudoscalar = reduce(lambda product, name: product * getattr(mv, name), names, one)
    return Bivector, Even, positives, pseudoscalar


# Three directions squaring to plus one, x y z, and one squaring to minus one, t.
Bivector, Even, positives, I = signature(3, 1)
print(Bivector.output_subspace)
print(Even.output_subspace)

## 1. The invariant form is the inner product

The generators of the rotors are the bivectors, and two of them combine by the commutator. The Lie algebra's invariant form takes two generators and traces what their double commutator does to a third: with all three left open, `Bivector.commutator(Bivector.commutator(Bivector))` has three bivector slots, and tracing out the last leaves a form with two. Built from types alone it is exact, in integers, and it comes out as `2 * (n - 2)` times the algebra's own inner product on bivectors, `Bivector | Bivector`: the geometric product carries the form already.

In the notation of Lie algebras this form reads as the Killing form, $K(a, b) = \operatorname{tr}(\operatorname{ad}_a \operatorname{ad}_b)$.

In [ ]:
form = Bivector.commutator(Bivector.commutator(Bivector)).trace(slot=2)        # [] Scalar <- (Bivector, Bivector)
inner = Bivector | Bivector                                                     # [] Scalar <- (Bivector, Bivector)
# Both are exact; subtracting four times the inner product leaves nothing.
difference = (form - 4 * inner).materialize(context)                            # [] Scalar <- (Bivector, Bivector)

In [ ]:
print(form)
print("largest entry of form - 4 * inner:", np.abs(difference.kernel).max())

## 2. Rotations and boosts

Sandwiching the bivectors with the product of the positive directions, `positives >> Bivector`, is an involution: applied twice it changes nothing. It negates exactly the planes that mix a positive with a negative direction. The planes it fixes square to minus one and generate rotations; the planes it negates square to plus one and generate boosts. With x y z positive and t negative, it fixes the three planes of x y z and negates the three planes that contain t.

In the notation of Lie algebras this involution reads as the Cartan involution, and its two eigenspaces as the Cartan decomposition $\mathfrak{k} \oplus \mathfrak{p}$; the group is compact when $\mathfrak{p}$ is empty.

In [ ]:
involution = positives >> Bivector                                               # [] Bivector <- Bivector
signs = involution.eigvals()                                                    # [planes] Scalar
# A plane of x y z and a plane containing t.
fixed, negated = involution(mv.xy), involution(mv.xt)                           # [] Bivector each

In [ ]:
print("eigenvalues:", np.real(signs.to_array()))
print("xy ->", render.blades(fixed), "   xt ->", render.blades(negated))

## 3. What the pseudoscalar does to spinors

In an even number of dimensions the pseudoscalar `I` of the chosen directions is itself an even multivector, and it commutes with every spinor. Multiplying by it, `I * Even`, is a map on spinors that commutes with every rotor, and it squares to `I * I`, plus or minus one. When that is plus one, its eigenvalues are plus and minus one, and `0.5 * (1 + I)` and `0.5 * (1 - I)` split the spinors into two halves that no rotor mixes. When it is minus one, its eigenvalues are plus and minus i: `I` acts on the spinors as the complex unit. Three positive directions with one negative are the second case; four positive directions are the first.

In matrix notation the spinors of the signature (3, 1), that of spacetime, read as pairs of complex numbers acted on by $SL(2,\mathbb{C})$, and those of four Euclidean dimensions as two independent pairs, one for each factor of $SU(2) \times SU(2)$.

In [ ]:
lorentzian = (I * Even).eigvals()                                                # [spinors] Scalar
_, EuclideanEven, _, euclidean_I = signature(4, 0)
euclidean = (euclidean_I * EuclideanEven).eigvals()                            # [spinors] Scalar

In [ ]:
print("signature (3, 1), I * I =", (I * I).select[0].to_array(), "  eigenvalues:", np.unique(np.round(lorentzian.to_array(), 6)))
print("signature (4, 0), I * I =", (euclidean_I * euclidean_I).select[0].to_array(), "  eigenvalues:", np.unique(np.round(euclidean.to_array(), 6)))

## 4. Two commuting halves in four dimensions

In four Euclidean directions the pseudoscalar maps bivectors to bivectors, so `plus` and `minus` split the generators too: `plus * generator` and `minus * generator` are bivectors, they commute, and the rotor is the product of their exponentials. Each of the two factors turns every plane through one and the same angle. The flow of each carries the points of the three-sphere around circles; through the stereographic projection every circle of one flow links every other once, and the two flows link in opposite senses. They are the Hopf fibration and its mirror image. Combining the two at different rates gives every other rotation: turning xy twice while zw turns three times is the flow against run five times over and the flow along run once backward, `5 * against - along`, and its orbits close only when both have come round, as trefoil knots on the tori.

In quaternion notation the two factors read as a pair of unit quaternions acting on a vector $v$ as $q_L\, v\, \bar q_R$.

In [ ]:
plus, minus = 0.5 * (1 + euclidean_I), 0.5 * (1 - euclidean_I)                 # [] Even each
# A generator turning in the planes xy, zw and xz at once, and its two halves.
generator = mv.xy * 0.4 + mv.zw * 0.9 + mv.xz * -0.3                              # [] Bivector
rotor = generator.exp()                                                         # [] Rotor
left, right = (plus * generator).exp(), (minus * generator).exp()               # [] Rotor each
Euclidean = ga.gatype(ga.subspace("x y z w"))
# The angles each factor turns its planes through.
left_turns = (left >> Euclidean).eigvals()                                      # [4] Scalar
right_turns = (right >> Euclidean).eigvals()                                    # [4] Scalar

In [ ]:
print("left * right - rotor:", np.abs((left * right - rotor).kernel).max(), "   left * right - right * left:", np.abs((left * right - right * left).kernel).max())
print("angles of left: ", np.abs(np.angle(left_turns.to_array())))
print("angles of right:", np.abs(np.angle(right_turns.to_array())))

In [ ]:

def stereographic(points):
    """Unit vectors of the four Euclidean directions, projected from -w into the space of x y z."""
    return (points - mv.w * (points | mv.w)) / (1 + (points | mv.w))


# The planes of the two flows: xy turned along with its dual plane zw, and against it.
along, against = plus * mv.xy, minus * mv.xy                                   # [] Bivector each
# Starting points at three heights along z, spread around the xy plane.
phi = np.linspace(0.0, 2 * np.pi, 12, endpoint=False)
heights = np.array([0.25, 0.6, 1.0])
# x turned up toward z by each height, then about z by each azimuth.
starts = (mv.xy * (-phi / 2)).exp() * (mv.xz * (-heights[:, None] / 2)).exp() >> mv.x   # [heights, azimuths] Vector
# One full turn of every plane: the sandwich turns by twice the rotor's angle.
turn = np.linspace(0.0, np.pi, 241)
left_orbits = stereographic((2 * along * turn).exp() >> starts[..., None])     # [heights, azimuths, turns] Space
right_orbits = stereographic((2 * against * turn).exp() >> starts[..., None])  # [heights, azimuths, turns] Space
# Turning xy twice while zw turns three times, from one start at each height.
combined = 2 * mv.xy + 3 * mv.zw                                                # [] Bivector
knotted_orbits = stereographic((combined * turn).exp() >> starts[:, :1, None])  # [heights, 1, turns] Space
render.draw_flows(left_orbits, right_orbits, knotted_orbits);

In [ ]:
def linking(first, second):
    """Gauss's linking number of two closed polygons in the space of x y z."""
    step_first = first[1:] - first[:-1]                                          # [n] Euclidean
    step_second = second[1:] - second[:-1]                                       # [m] Euclidean
    middle_first = 0.5 * (first[1:] + first[:-1])                                # [n] Euclidean
    middle_second = 0.5 * (second[1:] + second[:-1])                             # [m] Euclidean
    separation = middle_first[:, None] - middle_second[None, :]                  # [n, m] Euclidean
    # The trivector the separation spans with the two segments, measured against the unit volume xyz.
    volume = (separation ^ step_first[:, None] ^ step_second[None, :]) | mv.xyz.inverse()   # [n, m] Scalar
    return (volume / (separation | separation).square_root() ** 3).sum() / (4 * np.pi)   # [] Scalar


forward = linking(left_orbits[0, 0], left_orbits[2, 4])                         # [] Scalar
backward = linking(right_orbits[0, 0], right_orbits[2, 4])                      # [] Scalar
display(Image(filename=save_animation(render.animate_flows(left_orbits, right_orbits, knotted_orbits, 80), "spin_groups_isoclinic", 60)))

In [ ]:
print("linking of two orbits of the flow along:  ", forward.to_array())
print("linking of two orbits of the flow against:", backward.to_array())
print("combined - (5 * against - along):", render.blades(combined - (5 * against - along)) or "0")

## 5. Every signature up to six dimensions

The same three maps, for every choice of directions: how many planes there are and how many of them rotate or boost, the invariant form as a multiple of the inner product, the square of the pseudoscalar, and, in even dimensions, whether it splits the spinors or acts on them as the complex unit. In odd dimensions the pseudoscalar is not a spinor, and the spinors are one piece.

In [ ]:
signatures = ((3, 0), (2, 1), (4, 0), (3, 1), (2, 2), (5, 0), (4, 1), (3, 2), (6, 0), (5, 1), (4, 2), (3, 3))
rng = np.random.default_rng(0)


def rows():
    """The invariant form against the inner product on a sample bivector, the involution's
    eigenvalues, and the square of the pseudoscalar, for each signature."""
    for p, q in signatures:
        Bivector, Even, positives, pseudoscalar = signature(p, q)
        form = Bivector.commutator(Bivector.commutator(Bivector)).trace(slot=2).materialize(context)   # [] Scalar <- (Bivector, Bivector)
        sample = mv(Bivector.output_subspace, rng.normal(size=len(Bivector.output_subspace.masks)))   # [] Bivector
        yield p, q, form(sample, sample) / (sample | sample), (positives >> Bivector).eigvals(), (pseudoscalar * pseudoscalar).select[0]


def centres():
    """The pseudoscalar acting on the spinors, for each signature of an even number of dimensions."""
    for p, q in signatures[2:5] + signatures[8:]:
        _, Even, _, pseudoscalar = signature(p, q)
        yield p, q, (pseudoscalar * Even).eigvals()


print(render.table(rows(), centres()))

In the notation of Lie groups the rows read as $\mathrm{Spin}(3) = SU(2)$, $\mathrm{Spin}(2,1) = SL(2,\mathbb{R})$, $\mathrm{Spin}(4) = SU(2) \times SU(2)$, $\mathrm{Spin}(3,1) = SL(2,\mathbb{C})$, $\mathrm{Spin}(2,2) = SL(2,\mathbb{R}) \times SL(2,\mathbb{R})$, $\mathrm{Spin}(5) = Sp(2)$, $\mathrm{Spin}(4,1) = Sp(1,1)$, $\mathrm{Spin}(3,2) = Sp(4,\mathbb{R})$, $\mathrm{Spin}(6) = SU(4)$, $\mathrm{Spin}(5,1) = SL(2,\mathbb{H})$, $\mathrm{Spin}(4,2) = SU(2,2)$ and $\mathrm{Spin}(3,3) = SL(4,\mathbb{R})$.

In [ ]:
# checks
# The invariant form is four times the inner product in the signature (3, 1), exactly; the involution
# fixes the planes of x y z and negates those with t; the pseudoscalar squares to -1 in the signature
# (3, 1) and +1 in four Euclidean directions; the four-dimensional rotor is the product of its two commuting halves,
# each turning all its planes through one angle; the two flows link once, in opposite senses; and
# turning xy twice while zw turns three times is the flow against five times less the flow along.
np.testing.assert_array_equal(difference.kernel, 0.0)
np.testing.assert_allclose((fixed - mv.xy).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((negated + mv.xt).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose(np.abs(lorentzian.to_array()), 1.0, atol=1e-12)
np.testing.assert_allclose(np.abs(euclidean.to_array().imag), 0.0, atol=1e-12)
np.testing.assert_allclose((left * right - rotor).kernel, 0.0, atol=1e-8)
np.testing.assert_allclose((left * right - right * left).kernel, 0.0, atol=1e-12)
for turns in (left_turns, right_turns):
    angles = np.abs(np.angle(turns.to_array()))
    np.testing.assert_allclose(angles, angles[0], atol=1e-9)
np.testing.assert_allclose([forward.to_array(), -backward.to_array()], 1.0, atol=1e-2)
np.testing.assert_allclose((combined - (5 * against - along)).kernel, 0.0, atol=1e-12)